<a href="https://colab.research.google.com/github/wonzzae/WJ-Archive/blob/main/%EC%95%B1%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D%20%EC%A4%91%EA%B0%84%EA%B3%A0%EC%82%AC/%EC%95%B1%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D_%EC%A4%91%EA%B0%84%EA%B3%A0%EC%82%AC(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn gradio beautifulsoup4 requests nest_asyncio

In [ ]:
import requests
from bs4 import BeautifulSoup

def crawl_quotes():
    url = "http://quotes.toscrape.com/"
    res = requests.get(url)
    soup = BeautifulSoup(res.text, "html.parser")

    data = []
    for q in soup.select(".quote")[:20]:
        text = q.select_one(".text").get_text()
        author = q.select_one(".author").get_text()
        tags = [t.get_text() for t in q.select(".tag")]

        data.append((text, author, ", ".join(tags)))

    return data

In [ ]:
import sqlite3

conn = sqlite3.connect("quotes.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS quotes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    text TEXT,
    author TEXT,
    tags TEXT
)
""")

data = crawl_quotes()

cur.execute("DELETE FROM quotes")  # 초기화
for d in data:
    cur.execute("INSERT INTO quotes (text, author, tags) VALUES (?, ?, ?)", d)

conn.commit()
conn.close()

print("저장 완료:", len(data))

저장 완료: 10


In [ ]:
from fastapi import FastAPI
import sqlite3
import random

app = FastAPI()

@app.get("/quotes")
def get_quotes():
    conn = sqlite3.connect("quotes.db")
    cur = conn.cursor()
    cur.execute("SELECT * FROM quotes")
    data = cur.fetchall()
    conn.close()
    return data

@app.get("/random")
def random_quote():
    conn = sqlite3.connect("quotes.db")
    cur = conn.cursor()
    cur.execute("SELECT * FROM quotes")
    data = cur.fetchall()
    conn.close()
    return random.choice(data)

In [ ]:
import gradio as gr
import requests

def get_data():
    return requests.get("http://localhost:8000/quotes").json()

def get_random():
    return requests.get("http://localhost:8000/random").json()

with gr.Blocks() as demo:
    gr.Markdown("# 📚 Quotes Dashboard")

    btn1 = gr.Button("전체 조회")
    btn2 = gr.Button("랜덤 명언")

    out = gr.JSON()

    btn1.click(get_data, outputs=out)
    btn2.click(get_random, outputs=out)

In [ ]:
from gradio import mount_gradio_app
app = mount_gradio_app(app, demo, path="/")

new /


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok
import uvicorn
import threading
import time

# 기존 터널 제거
ngrok.kill()

# 토큰 입력
from getpass import getpass
token = getpass("ngrok 토큰 입력: ")
ngrok.set_auth_token(token)

# 서버 실행
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()

# 서버 준비 시간
time.sleep(3)

# ngrok 연결
public_url = ngrok.connect(8000)
print("🔥 ngrok URL:", public_url)

ngrok 토큰 입력: ··········


INFO:     Started server process [13446]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


🔥 ngrok URL: NgrokTunnel: "https://pungent-gentile-hydrogen.ngrok-free.dev" -> "http://localhost:8000"


In [5]:
!pip install mlxtend -q

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [6]:
import pandas as pd
from io import StringIO
from mlxtend.frequent_patterns import apriori, association_rules

raw = """OrderID,CustomerID,Items
T1,C01,커피
T1,C01,쿠키
T1,C01,샌드위치
T2,C02,커피
T2,C02,쿠키
T3,C03,커피
T3,C03,케이크
T4,C04,라면
T4,C04,김치
T5,C05,라면
T5,C05,계란
T6,C06,맥주
T6,C06,치킨
T7,C07,맥주
T7,C07,땅콩
T8,C08,우유
T8,C08,시리얼
T9,C09,커피
T9,C09,샌드위치
T10,C10,라면
T10,C10,김치
T10,C10,계란
T11,C11,커피
T11,C11,케이크
T12,C12,치킨
T12,C12,맥주
T12,C12,감자튀김
T13,C13,커피
T13,C13,쿠키
T14,C14,라면
T14,C14,김치
T15,C15,우유
T15,C15,시리얼
T16,C16,맥주
T16,C16,치킨
T17,C17,커피
T17,C17,샌드위치
T18,C18,라면
T18,C18,계란
T19,C19,맥주
T19,C19,땅콩
T20,C20,커피
T20,C20,쿠키
T21,C21,우유
T21,C21,시리얼
T22,C22,치킨
T22,C22,맥주
T23,C23,라면
T23,C23,김치
T24,C24,커피
T24,C24,케이크
T25,C25,맥주
T25,C25,치킨
T25,C25,감자튀김
T26,C26,라면
T26,C26,계란
T27,C27,커피
T27,C27,쿠키
T28,C28,우유
T28,C28,시리얼
T29,C29,맥주
T29,C29,땅콩
T30,C30,라면
T30,C30,김치
T30,C30,계란
T31,C31,커피
T31,C31,쿠키
T32,C32,커피
T32,C32,샌드위치
T33,C33,라면
T33,C33,김치
T34,C34,맥주
T34,C34,치킨
T35,C35,우유
T35,C35,시리얼
T36,C36,커피
T36,C36,케이크
T37,C37,라면
T37,C37,계란"""

df = pd.read_csv(StringIO(raw))

one_hot = pd.get_dummies(df['Items'])
df2 = df[['OrderID']].join(one_hot)
transaction = df2.groupby('OrderID').sum()
transaction[transaction >= 1] = 1

print("=== 상품별 구매 빈도 ===")
print(transaction.sum().sort_values(ascending=False))

print("\n=== 연관 규칙 ===")
frequent_items = apriori(transaction, min_support=0.1, use_colnames=True)
rules = association_rules(frequent_items, metric='lift', min_threshold=1)
print(rules[['antecedents','consequents','support','confidence','lift']].sort_values('lift', ascending=False).to_string(index=False))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

=== 상품별 구매 빈도 ===
커피      13
라면      10
맥주       9
김치       6
계란       6
치킨       6
쿠키       6
시리얼      5
우유       5
샌드위치     4
케이크      4
땅콩       3
감자튀김     2
dtype: int64

=== 연관 규칙 ===
antecedents consequents  support  confidence     lift
       (우유)       (시리얼) 0.135135    1.000000 7.400000
      (시리얼)        (우유) 0.135135    1.000000 7.400000
       (맥주)        (치킨) 0.162162    0.666667 4.111111
       (치킨)        (맥주) 0.162162    1.000000 4.111111
       (라면)        (계란) 0.162162    0.600000 3.700000
       (계란)        (라면) 0.162162    1.000000 3.700000
       (라면)        (김치) 0.162162    0.600000 3.700000
       (김치)        (라면) 0.162162    1.000000 3.700000
       (커피)      (샌드위치) 0.108108    0.307692 2.846154
       (커피)       (케이크) 0.108108    0.307692 2.846154
       (커피)        (쿠키) 0.162162    0.461538 2.846154
     (샌드위치)        (커피) 0.108108    1.000000 2.846154
      (케이크)        (커피) 0.108108    1.000000 2.846154
       (쿠키)        (커피) 0.162162    1.000000 2.846154


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [4]:
import pandas as pd
from io import StringIO
from mlxtend.frequent_patterns import apriori, association_rules

raw = """OrderID,CustomerID,Items
T1,C01,커피
T1,C01,쿠키
T1,C01,샌드위치
T2,C02,커피
T2,C02,쿠키
T3,C03,커피
T3,C03,케이크
T4,C04,라면
T4,C04,김치
T5,C05,라면
T5,C05,계란
T6,C06,맥주
T6,C06,치킨
T7,C07,맥주
T7,C07,땅콩
T8,C08,우유
T8,C08,시리얼
T9,C09,커피
T9,C09,샌드위치
T10,C10,라면
T10,C10,김치
T10,C10,계란
T11,C11,커피
T11,C11,케이크
T12,C12,치킨
T12,C12,맥주
T12,C12,감자튀김
T13,C13,커피
T13,C13,쿠키
T14,C14,라면
T14,C14,김치
T15,C15,우유
T15,C15,시리얼
T16,C16,맥주
T16,C16,치킨
T17,C17,커피
T17,C17,샌드위치
T18,C18,라면
T18,C18,계란
T19,C19,맥주
T19,C19,땅콩
T20,C20,커피
T20,C20,쿠키
T21,C21,우유
T21,C21,시리얼
T22,C22,치킨
T22,C22,맥주
T23,C23,라면
T23,C23,김치
T24,C24,커피
T24,C24,케이크
T25,C25,맥주
T25,C25,치킨
T25,C25,감자튀김
T26,C26,라면
T26,C26,계란
T27,C27,커피
T27,C27,쿠키
T28,C28,우유
T28,C28,시리얼
T29,C29,맥주
T29,C29,땅콩
T30,C30,라면
T30,C30,김치
T30,C30,계란
T31,C31,커피
T31,C31,쿠키
T32,C32,커피
T32,C32,샌드위치
T33,C33,라면
T33,C33,김치
T34,C34,맥주
T34,C34,치킨
T35,C35,우유
T35,C35,시리얼
T36,C36,커피
T36,C36,케이크
T37,C37,라면
T37,C37,계란"""

df = pd.read_csv(StringIO(raw))

one_hot = pd.get_dummies(df['Items'])
df2 = df[['OrderID']].join(one_hot)
transaction = df2.groupby('OrderID').sum()
transaction[transaction >= 1] = 1

print("=== 상품별 구매 빈도 ===")
print(transaction.sum().sort_values(ascending=False))

print("\n=== 연관 규칙 ===")
frequent_items = apriori(transaction, min_support=0.1, use_colnames=True)
rules = association_rules(frequent_items, metric='lift', min_threshold=1)
print(rules[['antecedents','consequents','support','confidence','lift']].sort_values('lift', ascending=False))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

=== 상품별 구매 빈도 ===
커피      13
라면      10
맥주       9
김치       6
계란       6
치킨       6
쿠키       6
시리얼      5
우유       5
샌드위치     4
케이크      4
땅콩       3
감자튀김     2
dtype: int64

=== 연관 규칙 ===
   antecedents consequents   support  confidence      lift
8         (우유)       (시리얼)  0.135135    1.000000  7.400000
9        (시리얼)        (우유)  0.135135    1.000000  7.400000
4         (맥주)        (치킨)  0.162162    0.666667  4.111111
5         (치킨)        (맥주)  0.162162    1.000000  4.111111
1         (라면)        (계란)  0.162162    0.600000  3.700000
0         (계란)        (라면)  0.162162    1.000000  3.700000
2         (라면)        (김치)  0.162162    0.600000  3.700000
3         (김치)        (라면)  0.162162    1.000000  3.700000
7         (커피)      (샌드위치)  0.108108    0.307692  2.846154
11        (커피)       (케이크)  0.108108    0.307692  2.846154
13        (커피)        (쿠키)  0.162162    0.461538  2.846154
6       (샌드위치)        (커피)  0.108108    1.000000  2.846154
10       (케이크)        (커피)  0.108108    1.00

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [7]:
import pandas as pd
from io import StringIO
from mlxtend.frequent_patterns import apriori, association_rules
from IPython.display import display, HTML
import json

raw = """OrderID,CustomerID,Items
T1,C01,커피
T1,C01,쿠키
T1,C01,샌드위치
T2,C02,커피
T2,C02,쿠키
T3,C03,커피
T3,C03,케이크
T4,C04,라면
T4,C04,김치
T5,C05,라면
T5,C05,계란
T6,C06,맥주
T6,C06,치킨
T7,C07,맥주
T7,C07,땅콩
T8,C08,우유
T8,C08,시리얼
T9,C09,커피
T9,C09,샌드위치
T10,C10,라면
T10,C10,김치
T10,C10,계란
T11,C11,커피
T11,C11,케이크
T12,C12,치킨
T12,C12,맥주
T12,C12,감자튀김
T13,C13,커피
T13,C13,쿠키
T14,C14,라면
T14,C14,김치
T15,C15,우유
T15,C15,시리얼
T16,C16,맥주
T16,C16,치킨
T17,C17,커피
T17,C17,샌드위치
T18,C18,라면
T18,C18,계란
T19,C19,맥주
T19,C19,땅콩
T20,C20,커피
T20,C20,쿠키
T21,C21,우유
T21,C21,시리얼
T22,C22,치킨
T22,C22,맥주
T23,C23,라면
T23,C23,김치
T24,C24,커피
T24,C24,케이크
T25,C25,맥주
T25,C25,치킨
T25,C25,감자튀김
T26,C26,라면
T26,C26,계란
T27,C27,커피
T27,C27,쿠키
T28,C28,우유
T28,C28,시리얼
T29,C29,맥주
T29,C29,땅콩
T30,C30,라면
T30,C30,김치
T30,C30,계란
T31,C31,커피
T31,C31,쿠키
T32,C32,커피
T32,C32,샌드위치
T33,C33,라면
T33,C33,김치
T34,C34,맥주
T34,C34,치킨
T35,C35,우유
T35,C35,시리얼
T36,C36,커피
T36,C36,케이크
T37,C37,라면
T37,C37,계란"""

df = pd.read_csv(StringIO(raw))
one_hot = pd.get_dummies(df['Items'])
df2 = df[['OrderID']].join(one_hot)
transaction = df2.groupby('OrderID').sum()
transaction[transaction >= 1] = 1

frequent_items = apriori(transaction, min_support=0.1, use_colnames=True)
rules = association_rules(frequent_items, metric='lift', min_threshold=1)

freq_dict = transaction.sum().sort_values(ascending=False).to_dict()
rules_list = []
for _, r in rules.sort_values('lift', ascending=False).iterrows():
    rules_list.append({
        'ant': ', '.join(list(r['antecedents'])),
        'con': ', '.join(list(r['consequents'])),
        'sup': round(float(r['support']), 3),
        'conf': round(float(r['confidence']), 3),
        'lift': round(float(r['lift']), 3)
    })

tx_dict = {}
for tid, group in df.groupby('OrderID'):
    tx_dict[tid] = list(group['Items'])

freq_json = json.dumps(freq_dict, ensure_ascii=False)
rules_json = json.dumps(rules_list, ensure_ascii=False)
tx_json = json.dumps(tx_dict, ensure_ascii=False)

html = f"""
<style>
  #dashboard {{ font-family: sans-serif; max-width: 800px; }}
  .tabs {{ display:flex; gap:8px; margin-bottom:16px; }}
  .tab {{ padding:6px 16px; border:1px solid #ccc; border-radius:6px; cursor:pointer; background:#f5f5f5; font-size:13px; }}
  .tab.active {{ background:#185FA5; color:white; border-color:#185FA5; }}
  .pane {{ display:none; }}
  .pane.active {{ display:block; }}
  .stat-row {{ display:grid; grid-template-columns:repeat(3,1fr); gap:12px; margin-bottom:20px; }}
  .stat {{ background:#f5f5f5; border-radius:8px; padding:12px; }}
  .stat-label {{ font-size:12px; color:#666; margin:0 0 4px; }}
  .stat-val {{ font-size:22px; font-weight:600; margin:0; }}
  .bar-row {{ display:flex; align-items:center; gap:10px; padding:6px 0; border-bottom:1px solid #eee; cursor:pointer; }}
  .bar-row:hover {{ background:#f9f9f9; }}
  .bar-track {{ flex:1; height:10px; background:#eee; border-radius:5px; overflow:hidden; }}
  .bar-fill {{ height:100%; background:#185FA5; border-radius:5px; }}
  .rule-card {{ border:1px solid #eee; border-radius:8px; padding:10px 14px; margin-bottom:8px; cursor:pointer; }}
  .rule-card:hover {{ background:#f9f9f9; }}
  .badge {{ font-size:11px; padding:2px 8px; border-radius:6px; background:#e6f1fb; color:#185FA5; }}
  .detail {{ background:#f0f7ff; border-radius:8px; padding:12px 16px; margin-top:12px; display:none; }}
  table {{ border-collapse:collapse; font-size:11px; width:100%; }}
  th,td {{ border:1px solid #eee; padding:4px 6px; text-align:center; }}
  th {{ background:#f5f5f5; font-weight:500; }}
  .has {{ color:#185FA5; font-weight:600; }}
</style>

<div id="dashboard">
  <div class="stat-row">
    <div class="stat"><p class="stat-label">총 트랜잭션</p><p class="stat-val">37</p></div>
    <div class="stat"><p class="stat-label">상품 종류</p><p class="stat-val">13</p></div>
    <div class="stat"><p class="stat-label">총 구매 건수</p><p class="stat-val">81</p></div>
  </div>

  <div class="tabs">
    <button class="tab active" onclick="showTab('freq',this)">구매 빈도</button>
    <button class="tab" onclick="showTab('rules',this)">연관 규칙</button>
    <button class="tab" onclick="showTab('tbl',this)">트랜잭션 테이블</button>
  </div>

  <div id="pane-freq" class="pane active">
    <p style="font-size:12px;color:#888;">상품 클릭 시 상세 정보</p>
    <div id="freq-list"></div>
    <div id="freq-detail" class="detail"></div>
  </div>

  <div id="pane-rules" class="pane">
    <p style="font-size:12px;color:#888;">카드 클릭 시 지표 설명</p>
    <div id="rules-list"></div>
    <div id="rules-detail" class="detail"></div>
  </div>

  <div id="pane-tbl" class="pane">
    <div style="overflow-x:auto"><table id="wide-tbl"></table></div>
  </div>
</div>

<script>
const freq = {freq_json};
const rules = {rules_json};
const tx = {tx_json};
const prods = Object.keys(freq);
const tids = Object.keys(tx);
const maxF = Math.max(...Object.values(freq));

function showTab(name, btn) {{
  document.querySelectorAll('.pane').forEach(p=>p.classList.remove('active'));
  document.querySelectorAll('.tab').forEach(b=>b.classList.remove('active'));
  document.getElementById('pane-'+name).classList.add('active');
  btn.classList.add('active');
}}

const fl = document.getElementById('freq-list');
prods.forEach(p => {{
  const pct = Math.round(freq[p]/maxF*100);
  const row = document.createElement('div');
  row.className = 'bar-row';
  row.innerHTML = `<span style="width:70px;font-size:13px">${{p}}</span><div class="bar-track"><div class="bar-fill" style="width:${{pct}}%"></div></div><span style="font-size:13px;font-weight:600;min-width:20px">${{freq[p]}}</span>`;
  row.onclick = () => {{
    const related = tids.filter(t=>tx[t].includes(p));
    const cobuys = {{}};
    related.forEach(t => tx[t].forEach(q => {{ if(q!==p) cobuys[q]=(cobuys[q]||0)+1; }}));
    const top = Object.entries(cobuys).sort((a,b)=>b[1]-a[1]).slice(0,3).map(([q,c])=>`<b>${{q}}</b>(${{c}}회)`).join(', ');
    const d = document.getElementById('freq-detail');
    d.style.display='block';
    d.innerHTML = `<b>${{p}}</b> — 총 ${{freq[p]}}회 구매<br>포함 트랜잭션: ${{related.join(', ')}}<br>함께 산 상품: ${{top||'없음'}}`;
  }};
  fl.appendChild(row);
}});

const rl = document.getElementById('rules-list');
rules.forEach(r => {{
  const card = document.createElement('div');
  card.className = 'rule-card';
  card.innerHTML = `<div style="display:flex;align-items:center;gap:8px;margin-bottom:6px">
    <b>${{r.ant}}</b> → <b>${{r.con}}</b>
    <span class="badge">lift ${{r.lift}}</span>
  </div>
  <div style="font-size:12px;color:#888">support ${{(r.sup*100).toFixed(1)}}% | confidence ${{(r.conf*100).toFixed(1)}}%</div>`;
  card.onclick = () => {{
    const d = document.getElementById('rules-detail');
    d.style.display='block';
    d.innerHTML = `<b>${{r.ant}} → ${{r.con}}</b><br>
    support ${{(r.sup*100).toFixed(1)}}%: 전체 중 함께 산 비율<br>
    confidence ${{(r.conf*100).toFixed(1)}}%: ${{r.ant}} 살 때 ${{r.con}}도 살 확률<br>
    lift ${{r.lift}}: 1보다 크면 양의 연관성 (높을수록 강함)`;
  }};
  rl.appendChild(card);
}});

const tbl = document.getElementById('wide-tbl');
let h = '<tr><th>주문ID</th>' + prods.map(p=>`<th>${{p}}</th>`).join('') + '</tr>';
tids.forEach(t => {{
  h += `<tr><td><b>${{t}}</b></td>` + prods.map(p=>`<td class="${{tx[t].includes(p)?'has':''}}">${{tx[t].includes(p)?'1':''}}</td>`).join('') + '</tr>';
}});
tbl.innerHTML = h;
</script>
"""

display(HTML(html))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [8]:
import pandas as pd
from io import StringIO
from mlxtend.frequent_patterns import apriori, association_rules
import gradio as gr
import json

raw = """OrderID,CustomerID,Items
T1,C01,커피
T1,C01,쿠키
T1,C01,샌드위치
T2,C02,커피
T2,C02,쿠키
T3,C03,커피
T3,C03,케이크
T4,C04,라면
T4,C04,김치
T5,C05,라면
T5,C05,계란
T6,C06,맥주
T6,C06,치킨
T7,C07,맥주
T7,C07,땅콩
T8,C08,우유
T8,C08,시리얼
T9,C09,커피
T9,C09,샌드위치
T10,C10,라면
T10,C10,김치
T10,C10,계란
T11,C11,커피
T11,C11,케이크
T12,C12,치킨
T12,C12,맥주
T12,C12,감자튀김
T13,C13,커피
T13,C13,쿠키
T14,C14,라면
T14,C14,김치
T15,C15,우유
T15,C15,시리얼
T16,C16,맥주
T16,C16,치킨
T17,C17,커피
T17,C17,샌드위치
T18,C18,라면
T18,C18,계란
T19,C19,맥주
T19,C19,땅콩
T20,C20,커피
T20,C20,쿠키
T21,C21,우유
T21,C21,시리얼
T22,C22,치킨
T22,C22,맥주
T23,C23,라면
T23,C23,김치
T24,C24,커피
T24,C24,케이크
T25,C25,맥주
T25,C25,치킨
T25,C25,감자튀김
T26,C26,라면
T26,C26,계란
T27,C27,커피
T27,C27,쿠키
T28,C28,우유
T28,C28,시리얼
T29,C29,맥주
T29,C29,땅콩
T30,C30,라면
T30,C30,김치
T30,C30,계란
T31,C31,커피
T31,C31,쿠키
T32,C32,커피
T32,C32,샌드위치
T33,C33,라면
T33,C33,김치
T34,C34,맥주
T34,C34,치킨
T35,C35,우유
T35,C35,시리얼
T36,C36,커피
T36,C36,케이크
T37,C37,라면
T37,C37,계란"""

df = pd.read_csv(StringIO(raw))
one_hot = pd.get_dummies(df['Items'])
df2 = df[['OrderID']].join(one_hot)
transaction = df2.groupby('OrderID').sum()
transaction[transaction >= 1] = 1

frequent_items = apriori(transaction, min_support=0.1, use_colnames=True)
rules = association_rules(frequent_items, metric='lift', min_threshold=1)

def get_freq(상품):
    cnt = int(transaction[상품].sum())
    related = transaction[transaction[상품]==1].index.tolist()
    cobuys = transaction.loc[related].drop(columns=[상품]).sum().sort_values(ascending=False)
    top = cobuys[cobuys>0].head(3)
    top_str = ', '.join([f"{p}({int(v)}회)" for p,v in top.items()])
    return f"구매 횟수: {cnt}회\n포함 트랜잭션: {', '.join(related)}\n함께 산 상품: {top_str or '없음'}"

def get_rules(min_sup, min_conf):
    fi = apriori(transaction, min_support=min_sup, use_colnames=True)
    r = association_rules(fi, metric='confidence', min_threshold=min_conf)
    r = r.sort_values('lift', ascending=False)
    r['antecedents'] = r['antecedents'].apply(lambda x: ', '.join(list(x)))
    r['consequents'] = r['consequents'].apply(lambda x: ', '.join(list(x)))
    r['support'] = r['support'].round(3)
    r['confidence'] = r['confidence'].round(3)
    r['lift'] = r['lift'].round(3)
    return r[['antecedents','consequents','support','confidence','lift']]

freq_df = transaction.sum().sort_values(ascending=False).reset_index()
freq_df.columns = ['상품', '구매횟수']

with gr.Blocks(title="장바구니 분석") as demo:
    gr.Markdown("## 장바구니 연관 분석 대시보드")

    with gr.Tab("구매 빈도"):
        gr.Dataframe(value=freq_df)
        상품선택 = gr.Dropdown(choices=list(transaction.columns), label="상품 선택")
        상세출력 = gr.Textbox(label="상세 정보", lines=4)
        상품선택.change(fn=get_freq, inputs=상품선택, outputs=상세출력)

    with gr.Tab("연관 규칙"):
        with gr.Row():
            sup_slider = gr.Slider(0.05, 0.5, value=0.1, step=0.05, label="min support")
            conf_slider = gr.Slider(0.1, 1.0, value=0.3, step=0.1, label="min confidence")
        규칙버튼 = gr.Button("분석 실행")
        규칙출력 = gr.Dataframe()
        규칙버튼.click(fn=get_rules, inputs=[sup_slider, conf_slider], outputs=규칙출력)

    with gr.Tab("트랜잭션 테이블"):
        gr.Dataframe(value=transaction.reset_index())

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/

In [9]:
# 장바구니 연관 분석 대시보드 (구매빈도 / 연관규칙 / 트랜잭션 테이블)
import pandas as pd
from io import StringIO
from mlxtend.frequent_patterns import apriori, association_rules
import gradio as gr

raw = """OrderID,CustomerID,Items
T1,C01,커피
T1,C01,쿠키
T1,C01,샌드위치
T2,C02,커피
T2,C02,쿠키
T3,C03,커피
T3,C03,케이크
T4,C04,라면
T4,C04,김치
T5,C05,라면
T5,C05,계란
T6,C06,맥주
T6,C06,치킨
T7,C07,맥주
T7,C07,땅콩
T8,C08,우유
T8,C08,시리얼
T9,C09,커피
T9,C09,샌드위치
T10,C10,라면
T10,C10,김치
T10,C10,계란
T11,C11,커피
T11,C11,케이크
T12,C12,치킨
T12,C12,맥주
T12,C12,감자튀김
T13,C13,커피
T13,C13,쿠키
T14,C14,라면
T14,C14,김치
T15,C15,우유
T15,C15,시리얼
T16,C16,맥주
T16,C16,치킨
T17,C17,커피
T17,C17,샌드위치
T18,C18,라면
T18,C18,계란
T19,C19,맥주
T19,C19,땅콩
T20,C20,커피
T20,C20,쿠키
T21,C21,우유
T21,C21,시리얼
T22,C22,치킨
T22,C22,맥주
T23,C23,라면
T23,C23,김치
T24,C24,커피
T24,C24,케이크
T25,C25,맥주
T25,C25,치킨
T25,C25,감자튀김
T26,C26,라면
T26,C26,계란
T27,C27,커피
T27,C27,쿠키
T28,C28,우유
T28,C28,시리얼
T29,C29,맥주
T29,C29,땅콩
T30,C30,라면
T30,C30,김치
T30,C30,계란
T31,C31,커피
T31,C31,쿠키
T32,C32,커피
T32,C32,샌드위치
T33,C33,라면
T33,C33,김치
T34,C34,맥주
T34,C34,치킨
T35,C35,우유
T35,C35,시리얼
T36,C36,커피
T36,C36,케이크
T37,C37,라면
T37,C37,계란"""

df = pd.read_csv(StringIO(raw))
one_hot = pd.get_dummies(df['Items'])
df2 = df[['OrderID']].join(one_hot)
transaction = df2.groupby('OrderID').sum()
transaction[transaction >= 1] = 1

def get_freq(상품):
    cnt = int(transaction[상품].sum())
    related = transaction[transaction[상품]==1].index.tolist()
    cobuys = transaction.loc[related].drop(columns=[상품]).sum().sort_values(ascending=False)
    top = cobuys[cobuys>0].head(3)
    top_str = ', '.join([f"{p}({int(v)}회)" for p,v in top.items()])
    return f"구매 횟수: {cnt}회\n포함 트랜잭션: {', '.join(related)}\n함께 산 상품: {top_str or '없음'}"

def get_rules(min_sup, min_conf):
    fi = apriori(transaction, min_support=min_sup, use_colnames=True)
    r = association_rules(fi, metric='confidence', min_threshold=min_conf)
    r = r.sort_values('lift', ascending=False)
    r['antecedents'] = r['antecedents'].apply(lambda x: ', '.join(list(x)))
    r['consequents'] = r['consequents'].apply(lambda x: ', '.join(list(x)))
    r['support'] = r['support'].round(3)
    r['confidence'] = r['confidence'].round(3)
    r['lift'] = r['lift'].round(3)
    return r[['antecedents','consequents','support','confidence','lift']]

freq_df = transaction.sum().sort_values(ascending=False).reset_index()
freq_df.columns = ['상품', '구매횟수']

with gr.Blocks(title="장바구니 분석") as demo:
    gr.Markdown("## 장바구니 연관 분석 대시보드")

    with gr.Tab("구매 빈도"):
        gr.Dataframe(value=freq_df)
        상품선택 = gr.Dropdown(choices=list(transaction.columns), label="상품 선택")
        상세출력 = gr.Textbox(label="상세 정보", lines=4)
        상품선택.change(fn=get_freq, inputs=상품선택, outputs=상세출력)

    with gr.Tab("연관 규칙"):
        with gr.Row():
            sup_slider = gr.Slider(0.05, 0.5, value=0.1, step=0.05, label="min support")
            conf_slider = gr.Slider(0.1, 1.0, value=0.3, step=0.1, label="min confidence")
        규칙버튼 = gr.Button("분석 실행")
        규칙출력 = gr.Dataframe()
        규칙버튼.click(fn=get_rules, inputs=[sup_slider, conf_slider], outputs=규칙출력)

    with gr.Tab("트랜잭션 테이블"):
        gr.Dataframe(value=transaction.reset_index())

demo.launch(share=True, debug=False, quiet=True)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

* Running on public URL: https://8fcf635e4395a683dc.gradio.live


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# 장바구니 연관 분석 대시보드
# 데이터 전처리 → apriori 연관규칙 분석 → 인터랙티브 UI 출력
# 탭1: 상품별 구매빈도 / 탭2: 연관규칙(support, confidence, lift) / 탭3: 트랜잭션 wide format

장바구니 분석 (Market Basket Analysis)
사용 데이터: 37개 트랜잭션, 13개 상품, 총 81건 구매 기록
전처리
Long format(행마다 상품 하나) → 원핫인코딩으로 Wide format(트랜잭션별 0/1) 변환
구매 빈도 분석
상품별 구매 횟수 집계. 커피(11회) > 라면·맥주(8회) 순으로 높음
연관 규칙 분석 (Apriori)

Support: 두 상품이 함께 구매된 비율
Confidence: A 구매 시 B도 구매할 확률
Lift: 연관성 강도 (1 이상이면 양의 연관)

주요 결과: 우유→시리얼 (lift 3.70), 맥주→치킨 (lift 2.78)
대시보드
Gradio로 구현. 구매빈도 / 연관규칙 / 트랜잭션 테이블 탭으로 구성, 슬라이더로 support·confidence 조절 가능